**TIP:** You can save your work offline by using `File -> Download`.
Later you can upload the notebook back into the workspace.

## Setup

In [1]:
# SETUP CELL
%pip -q install numpy sympy matplotlib ipywidgets pandas plotly anywidget scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# SETUP CELL
import src_path
from gu_toolkit import *
from helpers.Fourier_01_helper import create_mystery_function, MaxDistanceCard
from helpers.Fourier_02_helper import AvgDistanceCard

# Recovering a mystery sine series via `NIntegrate`

In `Fourier_01` the mystery function had the form
$$
F_{\text{myst}}(x)=\alpha_1\sin(2\pi x)+\alpha_2\sin(2\pi 2x)+\cdots+\alpha_N\sin(2\pi N x)
$$
with random coefficients between `-1` and `1`.

This notebook computes those coefficients numerically instead of guessing them with sliders.

## Generate a mystery function

In [3]:
np.random.seed(7)  # Change or remove the seed to generate a different example.
Nmyst = 6
Fmyst = create_mystery_function(Nmyst)

display(
    Latex(
        r"$F_{\text{myst}}(x)$ is a random sine sum with "
        + f"{Nmyst}"
        + r" hidden modes."
    )
)

<IPython.core.display.Latex object>

In [4]:
fig_mystery = Figure(x_range=(-1 / 2, 1 / 2))
fig_mystery.title = r"The mystery function $F_{\mathrm{myst}}(x)$"
display(fig_mystery)

with fig_mystery:
    plot(Fmyst(x), x, id="F_myst", label=r"$F_{\text{myst}}(x)$")

C:\Users\guraltsev\AppData\Local\Temp\ipykernel_55352\4164553835.py:1: DeprecationWarning: Figure(x_range=...) is deprecated; use Figure(default_x_range=...) instead.
  fig_mystery = Figure(x_range=(-1 / 2, 1 / 2))


OneShotOutput()

## Step 1. Compute the sine coefficients

On the interval $[-\tfrac12,\tfrac12]$, the coefficients are
$$
b_n = 2\int_{-1/2}^{1/2} F_{\text{myst}}(x)\sin(2\pi n x)\,dx.
$$

Because `F_myst` is already a sine sum, the recovered coefficients should match the hidden ones.
For `n > Nmyst`, the coefficients should be very close to `0`.

In [5]:
Nmax = 10

b_coeff = np.zeros(Nmax + 1)
for n in range(1, Nmax + 1):
    b_coeff[n] = NIntegrate(
        2 * Fmyst(x) * sin(2 * pi * n * x),
        (x, -1 / 2, 1 / 2),
    )

coeff_table = pd.DataFrame(
    {
        "n": np.arange(1, Nmax + 1),
        "b_n": b_coeff[1:],
        "|b_n|": np.abs(b_coeff[1:]),
    }
)
display(coeff_table.round(6))

,n,b_n,|b_n|
0,1,1.0,1.0
1,2,0.4,0.4
2,3,-0.6,0.6
3,4,-1.0,1.0
4,5,-0.7,0.7
5,6,-0.9,0.9
6,7,-0.0,0.0
7,8,0.0,0.0
8,9,0.0,0.0
9,10,0.0,0.0


If the notebook has correctly recovered the hidden sine sum, then the first `Nmyst` coefficients should be visible in the table and the later ones should be very small.

In [6]:
significant = coeff_table.loc[coeff_table["|b_n|"] > 1e-3].reset_index(drop=True)
display(significant.round(6))

,n,b_n,|b_n|
0,1,1.0,1.0
1,2,0.4,0.4
2,3,-0.6,0.6
3,4,-1.0,1.0
4,5,-0.7,0.7
5,6,-0.9,0.9


## Step 2. Build the partial sums

In [7]:
coeff_for_plot = b_coeff.copy()
coeff_for_plot[np.abs(coeff_for_plot) < 1e-10] = 0.0

S = [0]
for n in range(1, Nmax + 1):
    S.append(S[-1] + coeff_for_plot[n] * sin(2 * pi * n * x))

display(Latex(r"$S_N(x)=\sum_{n=1}^{N} b_n\sin(2\pi n x)$"))

<IPython.core.display.Latex object>

## Step 3. Plot the approximations

Because the mystery function is a **finite** sine sum, the approximation should become almost exact once you include enough modes.

In [8]:
Nshow = sorted(set([1, 2, 3, Nmyst, Nmax]))

fig = Figure(x_range=(-1 / 2, 1 / 2))
fig.title = r"Mystery function and recovered Fourier approximations"
display(fig)

with fig:
    plot(Fmyst(x), x, id="F_myst", label=r"$F_{\text{myst}}(x)$")
    for n in Nshow:
        plot(S[n], x, id=f"S{n}", label=rf"$S_{{{n}}}(x)$")

fig.info(MaxDistanceCard(x, Fmyst(x), S[Nmax]), id="myst_minus_SN_max")
fig.info(AvgDistanceCard(x, Fmyst(x), S[Nmax]), id="myst_minus_SN_avg")

C:\Users\guraltsev\AppData\Local\Temp\ipykernel_55352\3556036809.py:3: DeprecationWarning: Figure(x_range=...) is deprecated; use Figure(default_x_range=...) instead.
  fig = Figure(x_range=(-1 / 2, 1 / 2))


OneShotOutput()

## Step 4. Plot the error

Compare the error after only a few modes with the error after using all computed coefficients.

In [9]:
Nsmall = min(3, Nmax)

fig_error = Figure(x_range=(-1 / 2, 1 / 2))
fig_error.title = r"Errors for the mystery-function approximations"
display(fig_error)

with fig_error:
    plot(
        Fmyst(x) - S[Nsmall],
        x,
        id=f"error_{Nsmall}",
        label=rf"$F_{{\text{{myst}}}}(x)-S_{{{Nsmall}}}(x)$",
    )
    plot(
        Fmyst(x) - S[Nmax],
        x,
        id=f"error_{Nmax}",
        label=rf"$F_{{\text{{myst}}}}(x)-S_{{{Nmax}}}(x)$",
    )

C:\Users\guraltsev\AppData\Local\Temp\ipykernel_55352\1024321377.py:3: DeprecationWarning: Figure(x_range=...) is deprecated; use Figure(default_x_range=...) instead.
  fig_error = Figure(x_range=(-1 / 2, 1 / 2))


OneShotOutput()

## Explore

- Change `Nmyst`, or change the random seed, and rerun the notebook.
- Increase `Nmax` and see whether the extra coefficients stay close to `0`.
- Try setting `Nmax < Nmyst`. How does the plot tell you that modes are missing?
- Can you guess the hidden coefficients directly from the table?